In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
from torch import Tensor

In [3]:
import torch
from torch import nn
from torch import optim
import torch.utils.data as data
import math
import copy

# Multi-Head Attention

<img src="./images/linear.png" width=50%>

## Encoder

Hàm `scaled_dot_product` thực hiện
$$
\text{Attention}=\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

<img src="./images/encoder.png" heigh=50%>

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        if num_heads <= 0:
            raise ValueError("num_heads cannot less than equal zero")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model) # tạo ra W_q.weight, có shape (d_model, d_model) và bias (d_model,)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product(self, Q: Tensor, K: Tensor, V: Tensor, mask=None) -> Tensor:
        r"""
        Thực hiện tính toán
            Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
        """
        # Tính scores
        attn_scores = torch.matmul(Q, torch.transpose(K, -2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        # Sử dụng softmax
        attn_softmax = torch.softmax(attn_scores, dim=-1) # dim = index
        # Tính attention weight
        attn_weight = torch.matmul(attn_softmax, V)
        return attn_weight

    def split_heads(self, x : Tensor) -> Tensor:
        """
            x: Đại diện cho Q, K, V

            Cắt nhỏ chiều d_model thành num_heads phần bằng nhau, mỗi phần có kích thước là d_k (hay còn gọi là head_dim)
        """
        # Chia thành nhiều head để học
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) 

    def combie_heads(self, x : Tensor) -> Tensor:
        """
            x: attention output (softmax * V). Shape: [batch_size, num_heads, seq_length, d_k]
            
        """
        batch_size, _, seq_length, d_k = x.size()
        # [batch, num_heads, seq_len, d_k] →  [batch, seq_len, num_heads, d_k]. Mục đích: Đưa 2 chiều cần gộp là num_heads và d_k nằm cạnh nhau ở cuối tensor.
        # Contiguous: sắp xếp lại các phần tử trong bộ nhớ RAM/VRAM để chuẩn bị cho hàm view() (tránh lỗi bộ nhớ rời rạc sau hàm transpose).
        # .view(batch_size, seq_length, self.d_model): Gộp 2 chiều cuối (num_heads, d_k) lại thành một chiều duy nhất là d_model (vì theo định nghĩa: self.d_model = num_heads * d_k)
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q:Tensor, K:Tensor, V:Tensor, mask=None):
        # Nếu ở tầng đầu tiên: Là Word Embeddings + Positional Encoding của câu.
        # Nếu ở các tầng sâu hơn: Là kết quả đầu ra của tầng Transformer trước đó.
        # Ví dụ Q (Từ "nó")	→ W_q → Query (Câu hỏi) "Tìm danh từ/con vật đứng trước tôi."
        Q = self.split_heads(self.W_q(Q)) # Q = self.W_q.forward(x) tức là: Q = x @ self.W_q.weight.T + self.W_q.bias
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product(Q, K, V, mask) # Tính softmax * V
        output = self.W_o(self.combie_heads(attn_output))
        return output

In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        Đây chỉ đơn giản là 1 lớp fully connected layer
            d_model: Kích thước (chiều) đầu vào và đầu ra của mô hình
            d_ff: Kích thước của các lớp ẩn trong feed-forward network
        """
        super(FeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x : Tensor):
        return self.fc2(self.relu(self.fc1(x))) # layer 1 -> activation layer (relu) -> output layer

```
x.size()      # torch.Size([batch_size, seq_len, d_model])
x.size(0)     # batch_size  (int)
x.size(1)     # seq_len     (int)  <- đây là cái dùng trong code của bạn
x.size(2)     # d_model     (int)
```

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        """
            d_model: kích thước đầu vào của mô hình
        """
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) # chuyển về thành dạng ví dụ [[1], [2], ...]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() / d_model * -(math.log(100000))) # torch.arange = 2i
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0)) # đăng kí với buffer với key='pe', giá trị mẫu  [[1,2,3,4,...]] <- unsqueeze(0)

    def forward(self, x: Tensor):
        return x + self.pe[:, :x.size(1)] # Nó sử dụng x.size(1) phần tử đầu tiên của pe để đảm bảo rằng các mã hóa vị trí (positional encodings) tương ứng với độ dài thực tế của chuỗi x.
    


`self.self_attn` là một instance của `nn.Module` (cụ thể là MultiHeadAttention). Khi bạn gọi nó như một hàm — `self.self_attn(x, x, x)` — thực chất bạn đang gọi `self.self_attn.__call__(x, x, x)`, và `nn.Module.__call__` sẽ tự động gọi `forward()` bên trong.

In [7]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.05):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, mask=None):
        attn_output = self.self_attn(x, x, x, mask) # gọi như này là gọi trực tiếp hàm forward luôn
        x = self.norm1(x + self.dropout(attn_output)) # add and norm after multi-head
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

<img src="./images/decoder.png">

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [9]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # A list of encoder layers.
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # A list of decoder layers.

        self.fc = nn.Linear(d_model, tgt_vocab_size) # Final fully connected (linear) layer mapping to target vocabulary size
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device=tgt.device), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

# Training

In [12]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).to(device)

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)

In [13]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 8.676615715026855
Epoch: 2, Loss: 8.54863166809082
Epoch: 3, Loss: 8.481200218200684
Epoch: 4, Loss: 8.433833122253418
Epoch: 5, Loss: 8.383306503295898
Epoch: 6, Loss: 8.324868202209473
Epoch: 7, Loss: 8.243824005126953
Epoch: 8, Loss: 8.172118186950684
Epoch: 9, Loss: 8.083118438720703
Epoch: 10, Loss: 8.01229476928711
Epoch: 11, Loss: 7.933286190032959
Epoch: 12, Loss: 7.851909160614014
Epoch: 13, Loss: 7.766800880432129
Epoch: 14, Loss: 7.68137788772583
Epoch: 15, Loss: 7.602441787719727
Epoch: 16, Loss: 7.515544891357422
Epoch: 17, Loss: 7.438689708709717
Epoch: 18, Loss: 7.351513385772705
Epoch: 19, Loss: 7.272655963897705
Epoch: 20, Loss: 7.193715572357178
Epoch: 21, Loss: 7.126422882080078
Epoch: 22, Loss: 7.043006896972656
Epoch: 23, Loss: 6.959834575653076
Epoch: 24, Loss: 6.884186267852783
Epoch: 25, Loss: 6.813029766082764
Epoch: 26, Loss: 6.7297844886779785
Epoch: 27, Loss: 6.657134056091309
Epoch: 28, Loss: 6.582765102386475
Epoch: 29, Loss: 6.511767387390